# TSCT — Générateur de QCM (v1.0)

Colab wrapper around `scripts/run_qcm.py` from the public repo **fcavarretta/instest** — no logic lives here. Stdlib only, nothing to install.

**One-time setup** (🔑 Secrets panel, left sidebar — enable notebook access for each): `GEMINI_API_KEY` (required), and `GITHUB_TOKEN` (fine-grained token on `fcavarretta/instest` with **Contents: Read and write**) — **only needed to ⬆ PUSH in-class edits back**; reading/opening needs nothing, the repo is public.

**In class**: *Runtime → Run all*, then run the session cell pair you need. The repo is **⬇ pulled** into `/content/instest` (Setup section) — a full git working copy: browse and edit any YAML or prompt via the 📁 Files panel. **If you edited something, run the ⬆ PUSH cell at the bottom before leaving** — the Colab machine's disk is wiped when the runtime ends, and only pushed edits survive.

(The notebook file itself is the one thing ⬆ PUSH does not cover: after editing *this notebook*, use *File → Save a copy in GitHub*, same path, "Include a link to Colab" unchecked.)

## 1 · Setup — *Run all* passes through here; collapse afterwards (▸ next to this title)

In [ ]:
# @title Mount Google Drive (audio in, results out) { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title ⬇ PULL — get the latest from GitHub (start of session; never clobbers unpushed edits) { display-mode: "form" }
import os, subprocess

REPO = 'github.com/fcavarretta/instest.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
# Public repo: reading works without a token; the token (when present) enables ⬆ PUSH.
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'

def git(*args):
    return subprocess.run(['git', '-C', '/content/instest', *args], capture_output=True, text=True)

def show(r):
    out = r.stdout + r.stderr
    # Never print raw git output without masking — it can contain the tokenized URL.
    print(out.replace(token, '***') if token else out)
    assert r.returncode == 0, 'git failed — see output above'

if not os.path.exists('/content/instest'):
    show(subprocess.run(['git', 'clone', url, '/content/instest'], capture_output=True, text=True))
else:
    # Keep the remote in sync with the current token: granting the secret later
    # then re-running THIS cell is enough to make ⬆ PUSH work.
    git('remote', 'set-url', 'origin', url)
    if git('status', '--porcelain').stdout.strip():
        print('⚠️ Local edits not yet pushed to GitHub — skipping the pull so nothing is lost.')
        print('   Run the ⬆ PUSH cell at the bottom, then re-run this cell.')
    else:
        show(git('pull', '--ff-only'))

In [ ]:
# @title ⬆ Auto-PUSH — background send to GitHub every N min (live status below) { display-mode: "form" }
AUTOSAVE_MINUTES = 2  # @param {type:"integer"}
# The status line under this cell updates on every tick: ✅ + time of the last
# successful push, ⚠️ if anything failed. Safety net only — still run the ⬆ PUSH
# cell before leaving class. Interval changes apply after the current nap.
import threading, time, datetime
import ipywidgets as widgets
from IPython.display import display

_status = widgets.HTML(value='⏳ Auto-push armed — first check in a few minutes…')
display(_status)
_autosave_state = {'last_push': None, 'ok': True}

def _tick():
    now = datetime.datetime.now().strftime('%H:%M')
    if git('status', '--porcelain').stdout.strip():
        git('add', '-A')
        git('commit', '-m', f'Auto-save from Colab, {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
    # Push whenever local commits are ahead — also retries an earlier failed push.
    ahead = git('rev-list', '--count', '@{u}..HEAD').stdout.strip()
    if ahead not in ('', '0'):
        if git('push').returncode == 0:
            _autosave_state.update(last_push=now, ok=True)
            note = 'pushed'
        else:
            _autosave_state['ok'] = False
            note = 'PUSH FAILED — run the ⬆ PUSH cell'
    else:
        note = 'nothing to send'
    icon = '✅' if _autosave_state['ok'] else '⚠️'
    push_txt = f"last push <b>{_autosave_state['last_push']}</b>" if _autosave_state['last_push'] else 'no push yet'
    _status.value = f"{icon} {push_txt} · checked {now} ({note}) · every {AUTOSAVE_MINUTES} min"

def _autosave_loop():
    git('config', 'user.name', 'Fabrice Cavarretta (Colab)')
    git('config', 'user.email', 'fabrice@cavarretta.fr')
    while True:
        time.sleep(AUTOSAVE_MINUTES * 60)
        try:
            _tick()
        except Exception as e:
            _autosave_state['ok'] = False
            _status.value = f'⚠️ auto-push error: {e} — run the ⬆ PUSH cell'

if not any(t.name == 'tsct-autosave' for t in threading.enumerate()):
    threading.Thread(target=_autosave_loop, name='tsct-autosave', daemon=True).start()
else:
    _status.value = f'✅ Auto-push already running — interval now {AUTOSAVE_MINUTES} min (after current nap)'

## 2 · Sessions — one cell per session; run the one that just ended

In [ ]:
# @title Session 1 · step 1 — TRANSCRIBE (audio → X.transcript.md) { display-mode: "form" }
AUDIO = '/content/drive/MyDrive/_TSCT/Test/11-05Crop.m4a'  # @param {type:"string"}
# Hidden plumbing (edit here if ever needed, not form fields on purpose):
SESSION = '/content/instest/courses/DEMO/sessions/session-01.yaml'  # @param {type:"string"} # the session yaml — entry point of the parameter chain
OUTPUT = ''  # optional folder override; empty = outputs beside the audio
# Writes X.transcript.md beside the audio. Review it if you wish, then run step 2.

import sys
sys.path.insert(0, '/content/instest/scripts')
from run_qcm import main
args = [SESSION, '--audio', AUDIO, '--transcribe-only']
if OUTPUT.strip():
    args += ['--output-root', OUTPUT]
main(args)

In [ ]:
# @title Session 1 · step 2 — GENERATE (latest transcript → X.questions.gift) { display-mode: "form" }
# Uses AUDIO/OUTPUT/SESSION from step 1; picks the most recent transcript for that audio.
from pathlib import Path
import sys
sys.path.insert(0, '/content/instest/scripts')
from run_qcm import main
from lib.runfolder import find_latest_transcript

transcript = find_latest_transcript(Path(AUDIO), Path(OUTPUT) if OUTPUT.strip() else None)
print(f'Using transcript: {transcript}')
args = [SESSION, '--generate-only', '--transcript', str(transcript)]
if OUTPUT.strip():
    args += ['--output-root', OUTPUT]
main(args)

## 3 · ⬆ PUSH — send your work home before leaving (auto-push is only the net)

In [ ]:
# @title ⬆ PUSH — send your edits to GitHub (run before leaving class) { display-mode: "form" }
import datetime
git('config', 'user.name', 'Fabrice Cavarretta (Colab)')
git('config', 'user.email', 'fabrice@cavarretta.fr')
if not git('status', '--porcelain').stdout.strip():
    print('Nothing to send — working copy is clean.')
else:
    git('add', '-A')
    stamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    show(git('commit', '-m', f'In-class edits from Colab, {stamp}'))
    r = git('push')
    if r.returncode != 0:
        # Remote moved since we cloned (e.g. a push from home) — replay our commit on top.
        print('Push rejected — pulling remote changes and retrying…')
        p = git('pull', '--rebase')
        if p.returncode != 0:
            git('rebase', '--abort')
            print('❌ Same file changed on both sides — nothing lost (commit is local); copy this output to the AI.')
            print((p.stdout + p.stderr).replace(token, '***') if token else (p.stdout + p.stderr))
        else:
            r = git('push')
    if r.returncode == 0:
        print('✅ Pushed to GitHub — safe to close.')
    else:
        out = (r.stdout + r.stderr)
        print('❌ push failed:', out.replace(token, '***') if token else out)